# Claims SQL Playground

A hands-on SQL notebook built around a synthetic **healthcare claims**
dataset, going from basics to expert-level querying. It's modeled after the
kind of schema and question types that show up in data analytics / data
science interviews at healthcare payers (member enrollment, provider /
physician networks, claim adjudication, fraud triage).

Every query runs against [DuckDB](https://duckdb.org/), which lets us write
plain SQL directly against in-memory pandas DataFrames — no server, no setup.

## The schema

| Table | Grain (one row per...) | Notes |
|---|---|---|
| `members` | member | demographics, plan, enrollment |
| `providers` | facility / organization | hospital, clinic, pharmacy, etc. |
| `physicians` | physician | specialty, affiliated provider, **`supervisor_id`** (self-referencing, for hierarchy queries) |
| `diagnoses` | ICD-10 code | reference / lookup table |
| `procedures` | CPT code | reference / lookup table |
| `claims` | **claim header** | one row per claim: `claim_amount` / `allowed_amount` / `paid_amount`, `claim_status` |
| `claim_lines` | **claim line item** | 1-4 rows per claim — deliberately one-to-many, so we can practice spotting join fan-out bugs |

Run the setup cell once, then work through the sections top to bottom. Each
query is preceded by a short explanation and followed by **Pros / Cons**, so
you learn not just *how* to write it but *when* to reach for it.

## Table of contents
1. SQL Basics
2. Aggregation
3. Joins
4. Grain & Fan-Out Traps
5. Subqueries & CTEs
6. Date & String Functions
7. Window Functions
8. Expert Techniques
9. Fraud & Anomaly Detection
10. SQL vs. Python — choosing the right tool

In [1]:
import duckdb
import pandas as pd

import data_gen

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

con = duckdb.connect(database=":memory:")
tables = data_gen.load_into_duckdb(con)  # registers members, providers, physicians,
                                          # diagnoses, procedures, claims, claim_lines

for name, df in tables.items():
    print(f"{name:12s} {len(df):5d} rows")

providers        8 rows
physicians      20 rows
diagnoses       15 rows
procedures      15 rows
members        150 rows
claims        1000 rows
claim_lines   2431 rows


In [2]:
con.execute("SELECT * FROM claims LIMIT 5").fetchdf()

,claim_id,member_id,provider_id,physician_id,diagnosis_code,procedure_code,claim_type,service_date,submitted_date,claim_amount,allowed_amount,paid_amount,claim_status,denial_reason
0,1,113,6,16,I10,99213,Outpatient,2023-09-03,2023-09-08,931.81,867.95,867.95,Paid,None
1,2,128,3,10,I25.10,90834,Inpatient,2023-10-07,2023-10-20,500.81,389.57,305.28,Partially Paid,None
2,3,71,6,14,F41.1,59400,Pharmacy,2024-06-03,2024-06-11,2025.22,1923.18,1923.18,Paid,None
3,4,39,2,15,I25.10,93000,Outpatient,2023-12-20,2023-12-30,1281.85,853.75,853.75,Paid,None
4,5,147,1,13,M54.5,99396,Outpatient,2024-02-25,2024-02-26,683.01,646.09,646.09,Paid,None


## Part 1 — SQL Basics

### 1.1 SELECT & column aliasing

`SELECT` picks (and can compute) columns. `AS` renames them in the output.

In [3]:
con.execute('''
    SELECT
        member_id,
        first_name || ' ' || last_name AS full_name,
        plan_type,
        state
    FROM members
    LIMIT 5
''').fetchdf()

,member_id,full_name,plan_type,state
0,1,Sandra Williams,POS,MI
1,2,Carlos Whitney,EPO,TX
2,3,Jessica Thornton,POS,IL
3,4,Brian Landry,PPO,IL
4,5,Chase Johnson,PPO,IL


**Pros:** simple, declarative, reads like the question you're asking.
**Cons:** `SELECT *` is tempting but brittle in real pipelines — a schema
change upstream silently changes your output, and you pull columns (and
I/O) you don't need. Name columns explicitly outside of quick exploration.

### 1.2 WHERE — filtering rows

Comparison operators, `AND` / `OR` / `NOT`, `IN`, `BETWEEN`, `LIKE`.

In [4]:
con.execute('''
    SELECT claim_id, member_id, claim_status, claim_amount, service_date
    FROM claims
    WHERE claim_status IN ('Denied', 'Pending')
      AND claim_amount > 5000
      AND service_date BETWEEN DATE '2024-01-01' AND DATE '2024-06-30'
    ORDER BY claim_amount DESC
    LIMIT 10
''').fetchdf()

,claim_id,member_id,claim_status,claim_amount,service_date
0,255,30,Pending,9365.16,2024-03-03
1,735,45,Pending,8028.73,2024-02-10
2,796,25,Denied,5169.52,2024-05-03
3,933,147,Denied,5055.83,2024-03-23


**Pros:** filtering happens before grouping/sorting, so a selective `WHERE`
can drastically cut the data the engine has to touch.
**Cons:** wrapping a filtered column in a function (`WHERE UPPER(name) = ...`)
or a leading-wildcard `LIKE '%foo'` defeats the query planner's ability to
use statistics/indexes efficiently on a large, indexed table — not an issue
on this small in-memory dataset, but worth knowing for a real warehouse.

### 1.3 ORDER BY & LIMIT

Classic "top-N" reporting pattern.

In [5]:
con.execute('''
    SELECT claim_id, provider_id, claim_amount, claim_status
    FROM claims
    ORDER BY claim_amount DESC
    LIMIT 10
''').fetchdf()

,claim_id,provider_id,claim_amount,claim_status
0,324,3,18752.68,Paid
1,900,5,16652.60,Paid
2,780,5,14767.30,Paid
3,637,5,13186.06,Paid
4,622,2,13038.14,Pending
5,892,5,12726.26,Pending
6,995,3,9641.95,Pending
7,255,5,9365.16,Pending
8,437,7,9355.51,Paid
9,853,7,8942.77,Paid


**Pros:** combined with `LIMIT`, this is cheap even on huge tables — the
engine only has to track the top N, not sort everything.
**Cons:** `ORDER BY` with *no* `LIMIT` on a large result set is one of the
more expensive things you can ask a database to do. If you just need a
sorted view for eyeballing, sort the (much smaller) result in pandas/your BI
tool instead of sorting millions of rows in SQL.

### 1.4 DISTINCT

Removes duplicate rows from the result.

In [6]:
con.execute("SELECT DISTINCT claim_type, claim_status FROM claims ORDER BY 1, 2").fetchdf()

,claim_type,claim_status
0,Inpatient,Denied
1,Inpatient,Paid
2,Inpatient,Partially Paid
3,Inpatient,Pending
4,Outpatient,Denied
5,Outpatient,Paid
6,Outpatient,Partially Paid
7,Outpatient,Pending
8,Pharmacy,Denied
9,Pharmacy,Paid


**Pros:** a quick way to see the distinct values a column takes, or to
de-duplicate a result set.
**Cons:** reaching for `DISTINCT` to "fix" a result that has more rows than
expected is usually treating a symptom, not the cause — it often means
you've joined at the wrong grain (Part 4) and are now silently discarding
information instead of fixing the join. It's also not free: the engine has
to compare/sort every row to dedupe.

### 1.5 NULLs

`NULL` means "unknown" — it isn't equal to anything, including another
`NULL`. Our `Pending` claims have `paid_amount IS NULL` (not yet
adjudicated), which makes this dataset a good place to practice NULL
handling.

In [7]:
con.execute('''
    SELECT claim_id, claim_status, claim_amount, paid_amount,
           COALESCE(paid_amount, 0) AS paid_amount_or_zero
    FROM claims
    WHERE paid_amount IS NULL
    LIMIT 5
''').fetchdf()

,claim_id,claim_status,claim_amount,paid_amount,paid_amount_or_zero
0,13,Pending,8208.35,NaN,0.0
1,16,Pending,2724.49,NaN,0.0
2,21,Pending,922.84,NaN,0.0
3,38,Pending,326.86,NaN,0.0
4,53,Pending,1192.03,NaN,0.0


**Pros:** `COALESCE` gives an explicit, readable default instead of letting
`NULL` silently propagate through arithmetic (`NULL + 1 = NULL`) or
aggregates.
**Cons:** `WHERE paid_amount = NULL` is a classic beginner bug — it matches
*nothing*, ever; you must use `IS NULL` / `IS NOT NULL`. And blindly
`COALESCE`-ing to `0` can quietly change the meaning of an average or count
if "unknown" and "zero" aren't really the same thing for your question (a
pending claim isn't the same as a $0 claim).

## Part 2 — Aggregation

### 2.1 COUNT / SUM / AVG / MIN / MAX

In [8]:
con.execute('''
    SELECT
        COUNT(*)              AS n_claims,
        COUNT(paid_amount)    AS n_adjudicated,   -- COUNT(col) skips NULLs
        SUM(claim_amount)     AS total_billed,
        AVG(claim_amount)     AS avg_billed,
        MIN(claim_amount)     AS min_billed,
        MAX(claim_amount)     AS max_billed
    FROM claims
''').fetchdf()

,n_claims,n_adjudicated,total_billed,avg_billed,min_billed,max_billed
0,1000,898,1628131.77,1628.13177,81.35,18752.68


**Gotcha worth internalizing:** `COUNT(*)` counts rows; `COUNT(column)`
counts *non-NULL* values in that column. Here `n_adjudicated < n_claims`
because `Pending` claims have `paid_amount IS NULL` — that gap is itself a
useful metric (claims still awaiting adjudication).

### 2.2 GROUP BY

Collapses rows into groups and computes an aggregate per group. Every
non-aggregated column in `SELECT` must be in `GROUP BY` (or functionally
dependent on it) — that rule is the root of a lot of real-world SQL bugs.

In [9]:
con.execute('''
    SELECT
        claim_type,
        COUNT(*)          AS n_claims,
        SUM(claim_amount) AS total_billed,
        AVG(claim_amount) AS avg_billed
    FROM claims
    GROUP BY claim_type
    ORDER BY total_billed DESC
''').fetchdf()

,claim_type,n_claims,total_billed,avg_billed
0,Inpatient,256,444261.52,1735.396562
1,Pharmacy,255,438811.95,1720.831176
2,Professional,242,378850.74,1565.498926
3,Outpatient,247,366207.56,1482.621700


### 2.3 HAVING vs. WHERE

`WHERE` filters *rows* before grouping. `HAVING` filters *groups* after
aggregation. This is the single most common WHERE/HAVING mix-up: you cannot
say `WHERE SUM(paid_amount) > ...` because the aggregate doesn't exist yet
at the row-filtering stage.

In [10]:
con.execute('''
    SELECT
        p.provider_name,
        COUNT(*)               AS n_claims,
        SUM(c.paid_amount)     AS total_paid
    FROM claims c
    JOIN providers p ON p.provider_id = c.provider_id
    WHERE c.claim_status IN ('Paid', 'Partially Paid')   -- row filter, pre-aggregation
    GROUP BY p.provider_name
    HAVING SUM(c.paid_amount) > 500000                    -- group filter, post-aggregation
    ORDER BY total_paid DESC
''').fetchdf()

,provider_name,n_claims,total_paid


### 2.4 CASE WHEN for bucketing

`CASE` lets you compute a derived category inline, which you can then
`GROUP BY` like any other column. This pattern is the building block for the
fraud-flagging queries in Part 9.

In [11]:
con.execute('''
    SELECT
        CASE
            WHEN claim_amount < 500   THEN 'Low ( < $500 )'
            WHEN claim_amount < 5000  THEN 'Medium ( $500 - $5,000 )'
            ELSE 'High ( >= $5,000 )'
        END AS amount_tier,
        COUNT(*) AS n_claims
    FROM claims
    GROUP BY 1
    ORDER BY n_claims DESC
''').fetchdf()

,amount_tier,n_claims
0,"Medium ( $500 - $5,000 )",811
1,Low ( < $500 ),149
2,"High ( >= $5,000 )",40


**Pros (2.2 - 2.4):** aggregation is where SQL earns its keep — a `GROUP BY`
over millions of rows, computed by a columnar/vectorized engine, is usually
far faster than the equivalent loop in application code.
**Cons:** aggregation is lossy by definition — once you `GROUP BY`, row-level
detail is gone from that result set. If you need both the detail *and* a
group-level comparison on the same row, that's exactly what window
functions (Part 7) are for.

## Part 3 — Joins

### 3.1 INNER JOIN

Combines rows from two tables where the join condition matches. Rows with
no match on either side are dropped. Here we stitch together a full "claim
detail" view across five tables.

In [12]:
con.execute('''
    SELECT
        c.claim_id,
        m.first_name || ' ' || m.last_name AS member_name,
        pr.provider_name,
        ph.first_name || ' ' || ph.last_name AS physician_name,
        d.diagnosis_description,
        pc.procedure_description,
        c.claim_amount,
        c.claim_status
    FROM claims c
    JOIN members    m  ON m.member_id     = c.member_id
    JOIN providers  pr ON pr.provider_id  = c.provider_id
    JOIN physicians ph ON ph.physician_id = c.physician_id
    JOIN diagnoses  d  ON d.diagnosis_code = c.diagnosis_code
    JOIN procedures pc ON pc.procedure_code = c.procedure_code
    LIMIT 10
''').fetchdf()

,claim_id,member_name,provider_name,physician_name,diagnosis_description,procedure_description,claim_amount,claim_status
0,1,Tiffany Vaughn,"Zuniga, Wong and Lynch Health Group",Christine Barnes,Essential (primary) hypertension,"Office/outpatient visit, established patient, ...",931.81,Paid
1,2,Terry Evans,"Henderson, Ramirez and Lewis Medical Center",Michelle Brown,Atherosclerotic heart disease of native corona...,"Psychotherapy, 45 minutes",500.81,Partially Paid
2,3,Thomas Hooper,"Zuniga, Wong and Lynch Health Group",Beth Daniels,Generalized anxiety disorder,"Routine obstetric care, vaginal delivery",2025.22,Paid
3,4,Rebecca Moyer,Wagner Inc Medical Center,Christopher Lopez,Atherosclerotic heart disease of native corona...,"Electrocardiogram, routine",1281.85,Paid
4,5,Samantha Johnston,"Rodriguez, Figueroa and Sanchez Health Group",Tricia Baker,Low back pain,"Periodic preventive exam, established patient,...",683.01,Paid
5,6,Luke Rowe,Wagner Inc Medical Center,Wendy Rice,"Unspecified asthma, uncomplicated",Total knee replacement,2745.04,Paid
6,7,Mary Campbell,Galloway-Wyatt Health Group,Shawn Mckay,"Fracture of lower end of radius, initial encou...","Injection, ketorolac tromethamine",1452.09,Paid
7,8,Ashley Gould,"Zuniga, Wong and Lynch Health Group",Christine Barnes,General adult medical examination,Total knee replacement,1994.85,Paid
8,9,Brittany Williams,"Henderson, Ramirez and Lewis Medical Center",Michelle Brown,Encounter for full-term uncomplicated delivery,"Office/outpatient visit, established patient, ...",1949.19,Paid
9,10,Logan Mack,"Stevens, Martinez and Nielsen Health Group",Joseph Brennan,Essential (primary) hypertension,Knee arthroscopy with meniscectomy,1808.66,Paid


**Pros:** this is the workhorse of relational data — related facts live in
narrow, non-redundant tables, and you join them back together only when you
need the combined view. **Cons:** every join is a potential grain change; the
moment any one of these tables isn't 1-to-1 with `claims`, an `INNER JOIN`
can silently multiply or drop rows (see Part 4).

### 3.2 LEFT JOIN — the "find what's missing" pattern

`LEFT JOIN` keeps every row from the left table, filling in `NULL` where
there's no match on the right. Filtering on `right.key IS NULL` afterward
gives you an **anti-join**: rows on the left with *no* match at all.

In [13]:
con.execute('''
    SELECT m.member_id, m.first_name, m.last_name, m.enrollment_date
    FROM members m
    LEFT JOIN claims c ON c.member_id = m.member_id
    WHERE c.claim_id IS NULL
    ORDER BY m.enrollment_date
''').fetchdf()

,member_id,first_name,last_name,enrollment_date


**Pros:** this is the standard way to answer "which members have never filed
a claim" / "which procedures were never billed" style utilization questions
— you cannot get this from `INNER JOIN`. **Cons:** easy to get backwards
(`RIGHT JOIN` vs `LEFT JOIN`, or forgetting the `IS NULL` filter and instead
getting *every* member with their claims, most with real claim data mixed
in) — always sanity-check row counts against `COUNT(*) FROM members`.

### 3.3 Self-join

Joins a table to itself. `physicians.supervisor_id` points back to another
`physicians.physician_id`, so we can pair each physician with their direct
supervisor.

In [14]:
con.execute('''
    SELECT
        p.first_name || ' ' || p.last_name AS physician,
        p.specialty,
        s.first_name || ' ' || s.last_name AS supervisor
    FROM physicians p
    JOIN physicians s ON p.supervisor_id = s.physician_id
    ORDER BY supervisor, physician
''').fetchdf()

,physician,specialty,supervisor
0,Eric Campbell,Radiology,Christine Barnes
1,Nathaniel Martin,Radiology,Christine Barnes
2,Beth Daniels,Internal Medicine,Jason Baker
3,Brian Humphrey,Internal Medicine,Jason Baker
4,Sara Allison,Dermatology,Jason Baker
5,Shawn Mckay,Cardiology,Jason Baker
6,Christopher Lopez,Family Medicine,Wendy Rice
7,David Brewer,Psychiatry,Wendy Rice
8,Joseph Brennan,Cardiology,Wendy Rice
9,Barbara Sanchez,General Surgery,William Davis


**Pros:** simple and fast for a single level of a hierarchy (employee ->
manager, physician -> supervisor). **Cons:** a self-join only walks *one*
level. To get the full chain up to the top (arbitrary depth), you need a
recursive CTE — see Part 8.

## Part 4 — Grain & Fan-Out Traps

This is the bug that shows up most often in real analytics work (and in
"debug this query" interview rounds): joining to a table with a finer grain
than you think, then aggregating, and getting numbers that are quietly
*too big*.

`claim_lines` has 1-4 rows per claim. `claims.claim_amount` is a
**claim-header** total. Watch what happens when we join the two and sum
`claim_amount`.

In [15]:
# THE BUG: claim_amount belongs to the claim header (grain = 1 row/claim),
# but joining to claim_lines duplicates that header value once per line.
wrong = con.execute('''
    SELECT c.provider_id, SUM(c.claim_amount) AS total_wrong
    FROM claims c
    JOIN claim_lines cl ON cl.claim_id = c.claim_id
    GROUP BY c.provider_id
    ORDER BY c.provider_id
''').fetchdf()

correct = con.execute('''
    SELECT provider_id, SUM(claim_amount) AS total_correct
    FROM claims
    GROUP BY provider_id
    ORDER BY provider_id
''').fetchdf()

compare = wrong.merge(correct, on='provider_id')
compare['inflation_pct'] = ((compare['total_wrong'] / compare['total_correct']) - 1) * 100
compare

,provider_id,total_wrong,total_correct,inflation_pct
0,1,187433.71,77364.82,142.272534
1,2,422323.22,164799.54,156.264805
2,3,659465.88,248275.33,165.618771
3,4,661590.26,296824.94,122.889040
4,5,1262466.39,514832.83,145.218703
5,6,339458.11,157108.28,116.066340
6,7,422109.21,168926.03,149.878133


Every provider's total is inflated — by roughly however many line items
their claims average, because each claim's header amount got repeated once
per line row before the `SUM`. Nothing in the query *looked* wrong; the bug
is purely about grain.

**The fix:** either don't join to the finer-grained table when you don't
need it (the `correct` query above), or, if you *do* need line-item detail,
pre-aggregate the child table to the parent's grain in a CTE first, so the
join is 1-to-1:

In [16]:
con.execute('''
    WITH line_totals AS (
        SELECT claim_id, COUNT(*) AS n_lines, SUM(line_charge_amount) AS lines_total
        FROM claim_lines
        GROUP BY claim_id
    )
    SELECT
        c.provider_id,
        SUM(c.claim_amount) AS header_total,      -- correct: claims is still 1 row/claim here
        SUM(lt.lines_total) AS line_level_total    -- for comparison; should be ~equal, not a multiple
    FROM claims c
    JOIN line_totals lt ON lt.claim_id = c.claim_id
    GROUP BY c.provider_id
    ORDER BY c.provider_id
''').fetchdf()

,provider_id,header_total,line_level_total
0,1,77364.82,77364.78
1,2,164799.54,164799.53
2,3,248275.33,248275.29
3,4,296824.94,296825.00
4,5,514832.83,514832.91
5,6,157108.28,157108.25
6,7,168926.03,168926.05


**Pros of knowing this pattern:** it's a five-minute fix once you spot it,
and spotting it fast (rather than shipping a dashboard with 2-3x inflated
totals) is a real, high-leverage skill. **Cons / no silver bullet:** there's
no syntax that automatically protects you from this — the only defense is
habitually asking **"what is the grain of each table in this join, and does
it match what I'm about to aggregate?"** before you hit run. `SELECT
DISTINCT` or `SUM(DISTINCT ...)` are sometimes offered as a quick patch —
resist that urge; they mask the symptom (and can drop legitimately distinct
rows) instead of fixing the join.

## Part 5 — Subqueries & CTEs

### 5.1 Scalar subquery

A subquery that returns a single value can be used anywhere a literal could
go — here, inline in a `WHERE` clause.

In [17]:
con.execute('''
    SELECT claim_id, provider_id, claim_amount
    FROM claims
    WHERE claim_amount > (SELECT AVG(claim_amount) FROM claims)
    ORDER BY claim_amount DESC
    LIMIT 10
''').fetchdf()

,claim_id,provider_id,claim_amount
0,324,3,18752.68
1,900,5,16652.60
2,780,5,14767.30
3,637,5,13186.06
4,622,2,13038.14
5,892,5,12726.26
6,995,3,9641.95
7,255,5,9365.16
8,437,7,9355.51
9,853,7,8942.77


### 5.2 Correlated subquery

A subquery that references a column from the *outer* query, so it's
re-evaluated per outer row. Here: claims priced above **their own
provider's** average, not the global average.

In [18]:
con.execute('''
    SELECT c1.claim_id, c1.provider_id, c1.claim_amount
    FROM claims c1
    WHERE c1.claim_amount > (
        SELECT AVG(c2.claim_amount)
        FROM claims c2
        WHERE c2.provider_id = c1.provider_id   -- correlation to the outer row
    )
    ORDER BY c1.provider_id, c1.claim_amount DESC
    LIMIT 10
''').fetchdf()

,claim_id,provider_id,claim_amount
0,230,1,7101.41
1,246,1,6110.32
2,63,1,3985.82
3,507,1,3800.64
4,942,1,3132.50
5,699,1,3035.93
6,939,1,2915.93
7,665,1,2766.39
8,252,1,2698.34
9,620,1,2290.60


**Pros:** correlated subqueries are very readable for "compare this row to
its own group" questions. **Cons:** conceptually (and often literally) the
engine re-runs the inner query once per outer row — Part 7 shows how a
window function (`AVG(...) OVER (PARTITION BY ...)`) computes the exact same
thing in a single pass, and keeps every row instead of just the ones that
pass the filter.

### 5.3 IN vs. EXISTS

Both answer "does a matching row exist elsewhere?" — but they behave
differently, especially around `NULL`s.

In [19]:
in_version = con.execute('''
    SELECT member_id, first_name, last_name
    FROM members
    WHERE member_id IN (SELECT member_id FROM claims WHERE claim_status = 'Denied')
    ORDER BY member_id
''').fetchdf()

exists_version = con.execute('''
    SELECT m.member_id, m.first_name, m.last_name
    FROM members m
    WHERE EXISTS (
        SELECT 1 FROM claims c
        WHERE c.member_id = m.member_id AND c.claim_status = 'Denied'
    )
    ORDER BY m.member_id
''').fetchdf()

print(in_version.equals(exists_version))
in_version.head()

True


,member_id,first_name,last_name
0,2,Carlos,Whitney
1,3,Jessica,Thornton
2,7,Holly,Fitzpatrick
3,8,Amanda,Howard
4,10,Shaun,Dickson


**Pros:** `EXISTS` short-circuits (stops at the first match) and is safe
even if the subquery's column can contain `NULL`s. **Cons:** `NOT IN` with a
subquery that can return `NULL` is a classic trap — if even one `NULL`
sneaks into the `IN` list, the whole `NOT IN` matches *nothing*, for every
row, with no error. `NOT EXISTS` doesn't have this problem, so prefer it
over `NOT IN` whenever the subquery's column is nullable.

### 5.4 CTEs (`WITH`) — the readable rewrite

A CTE is a named, temporary result set you can reference later in the same
query. It doesn't change *what* is computed here — it's the same logic as
5.2 — but it reads top-to-bottom instead of nesting outward-in.

In [20]:
con.execute('''
    WITH provider_avg AS (
        SELECT provider_id, AVG(claim_amount) AS avg_claim_amount
        FROM claims
        GROUP BY provider_id
    )
    SELECT c.claim_id, c.provider_id, c.claim_amount, pa.avg_claim_amount
    FROM claims c
    JOIN provider_avg pa ON pa.provider_id = c.provider_id
    WHERE c.claim_amount > pa.avg_claim_amount
    ORDER BY c.provider_id, c.claim_amount DESC
    LIMIT 10
''').fetchdf()

,claim_id,provider_id,claim_amount,avg_claim_amount
0,230,1,7101.41,1432.681852
1,246,1,6110.32,1432.681852
2,63,1,3985.82,1432.681852
3,507,1,3800.64,1432.681852
4,942,1,3132.50,1432.681852
5,699,1,3035.93,1432.681852
6,939,1,2915.93,1432.681852
7,665,1,2766.39,1432.681852
8,252,1,2698.34,1432.681852
9,620,1,2290.60,1432.681852


**Pros:** far more readable once queries get to 3+ logical steps, and a CTE
can be reused multiple times later in the same query without repeating
logic. **Cons:** the readability win is real, but don't assume a CTE is a
free performance optimization — behavior varies by engine/version (DuckDB's
optimizer can inline and reorder them; some older engines, e.g. Postgres
before v12, always materialized a CTE, which could be *slower* than the
equivalent subquery). Know your engine if you're tuning for performance, not
just readability.

### 5.5 Chaining multiple CTEs

CTEs can reference earlier CTEs, letting you build a pipeline of named
steps. This is the structure we'll reuse heavily in Part 9.

In [21]:
con.execute('''
    WITH provider_stats AS (
        SELECT provider_id, AVG(claim_amount) AS avg_claim_amount, COUNT(*) AS n_claims
        FROM claims
        GROUP BY provider_id
    ),
    flagged AS (
        SELECT c.claim_id, c.provider_id, c.claim_amount, ps.avg_claim_amount,
               CASE WHEN c.claim_amount > 2 * ps.avg_claim_amount THEN 1 ELSE 0 END AS is_high_outlier
        FROM claims c
        JOIN provider_stats ps ON ps.provider_id = c.provider_id
    )
    SELECT provider_id, COUNT(*) AS n_flagged
    FROM flagged
    WHERE is_high_outlier = 1
    GROUP BY provider_id
    ORDER BY n_flagged DESC
''').fetchdf()

,provider_id,n_flagged
0,5,31
1,4,20
2,3,14
3,7,11
4,6,11
5,2,9
6,1,7


## Part 6 — Date & String Functions

### 6.1 Date arithmetic & truncation

Subtracting two `DATE` columns gives you a day count directly. `DATE_TRUNC`
buckets a date down to a month (or year, week, etc.) for trend reporting.

In [22]:
con.execute('''
    SELECT
        claim_id,
        service_date,
        submitted_date,
        submitted_date - service_date AS days_to_submit
    FROM claims
    ORDER BY days_to_submit DESC
    LIMIT 10
''').fetchdf()

,claim_id,service_date,submitted_date,days_to_submit
0,14,2023-09-14,2023-10-05,21
1,121,2024-05-09,2024-05-30,21
2,99,2023-03-05,2023-03-26,21
3,42,2024-04-04,2024-04-25,21
4,155,2024-06-10,2024-07-01,21
5,34,2023-02-20,2023-03-13,21
6,161,2024-01-16,2024-02-06,21
7,88,2023-11-29,2023-12-20,21
8,67,2023-06-07,2023-06-28,21
9,199,2023-04-01,2023-04-22,21


In [23]:
con.execute('''
    SELECT
        DATE_TRUNC('month', service_date) AS service_month,
        COUNT(*) AS n_claims,
        SUM(claim_amount) AS total_billed
    FROM claims
    GROUP BY 1
    ORDER BY 1
''').fetchdf()

,service_month,n_claims,total_billed
0,2023-01-01,58,131529.21
1,2023-02-01,49,82917.85
2,2023-03-01,57,91288.29
3,2023-04-01,69,134393.92
4,2023-05-01,54,75606.39
5,2023-06-01,51,90622.35
6,2023-07-01,57,87779.96
7,2023-08-01,49,78055.77
8,2023-09-01,53,69554.75
9,2023-10-01,59,86953.60


### 6.2 String functions

`||` concatenates, `UPPER`/`LOWER` normalize case, `SUBSTR`/`LEFT` slice
strings — handy for grouping by, e.g., the ICD-10 chapter (first letter of
the diagnosis code).

In [24]:
con.execute('''
    SELECT
        LEFT(c.diagnosis_code, 1) AS icd_chapter_letter,
        COUNT(*) AS n_claims
    FROM claims c
    JOIN diagnoses d ON d.diagnosis_code = c.diagnosis_code
    GROUP BY 1
    ORDER BY n_claims DESC
''').fetchdf()

,icd_chapter_letter,n_claims
0,J,153
1,E,144
2,I,130
3,M,114
4,S,71
5,R,71
6,N,70
7,F,69
8,K,69
9,O,58


**Pros (6.1 - 6.2):** date/string functions let you derive new analytical
dimensions (month, chapter, category) without touching the source data.
**Cons:** applying a function to a *filtered or joined* column
(`WHERE LEFT(diagnosis_code, 1) = 'E'` on a large indexed table) generally
prevents the engine from using an index on that column — if that predicate
matters for performance at scale, consider storing the derived value as its
own column instead of computing it at query time.

## Part 7 — Window Functions

A window function computes something *across a group of rows* (like an
aggregate) but **without collapsing them** — every input row survives in the
output, now carrying a group-level fact alongside it. This is the tool that
replaces the correlated subquery from 5.2 and sets up everything in Part 9.

General form: `<function>(...) OVER (PARTITION BY <group> ORDER BY <order>)`.

### 7.1 ROW_NUMBER vs. RANK vs. DENSE_RANK

All three number rows within a partition by some order — they only differ
on how they handle **ties**.

In [25]:
con.execute('''
    WITH physician_totals AS (
        SELECT physician_id, SUM(paid_amount) AS total_paid
        FROM claims
        WHERE paid_amount IS NOT NULL
        GROUP BY physician_id
    )
    SELECT
        physician_id,
        total_paid,
        ROW_NUMBER() OVER (ORDER BY total_paid DESC) AS row_num,     -- always unique, arbitrary tiebreak
        RANK()       OVER (ORDER BY total_paid DESC) AS rnk,         -- ties share a rank, next rank skips
        DENSE_RANK() OVER (ORDER BY total_paid DESC) AS dense_rnk    -- ties share a rank, next rank doesn't skip
    FROM physician_totals
    ORDER BY total_paid DESC
    LIMIT 10
''').fetchdf()

,physician_id,total_paid,row_num,rnk,dense_rnk
0,19,80016.38,1,1,1
1,10,65946.37,2,2,2
2,7,60462.04,3,3,3
3,20,52947.92,4,4,4
4,14,50033.17,5,5,5
5,4,49712.59,6,6,6
6,15,46816.71,7,7,7
7,18,43842.21,8,8,8
8,6,41937.90,9,9,9
9,16,41093.54,10,10,10


**Pros:** precise control over tie behavior — `ROW_NUMBER` for "give me
exactly one," `RANK`/`DENSE_RANK` for "give me an honest leaderboard where
ties are ties." **Cons:** it's easy to grab `ROW_NUMBER` out of habit when
you actually want `RANK` (e.g., "top 3 physicians by volume" should probably
include a 4th physician tied for 3rd — `ROW_NUMBER` would arbitrarily cut
one of them).

### 7.2 The "latest record per group" pattern

An extremely common real-world need: one row per member, but only their
*most recent* claim. `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ... DESC)`
then filtering to `rn = 1` is the standard way to do it.

In [26]:
con.execute('''
    WITH ranked AS (
        SELECT
            claim_id, member_id, service_date, claim_amount, claim_status,
            ROW_NUMBER() OVER (PARTITION BY member_id ORDER BY service_date DESC) AS rn
        FROM claims
    )
    SELECT claim_id, member_id, service_date, claim_amount, claim_status
    FROM ranked
    WHERE rn = 1
    ORDER BY member_id
    LIMIT 10
''').fetchdf()

,claim_id,member_id,service_date,claim_amount,claim_status
0,578,1,2024-05-08,2532.08,Paid
1,886,2,2024-05-15,1089.80,Paid
2,913,3,2023-12-09,1374.11,Paid
3,376,4,2024-05-25,3217.56,Paid
4,576,5,2023-04-14,3089.55,Paid
5,80,6,2024-04-13,599.36,Paid
6,626,7,2024-04-17,767.22,Paid
7,203,8,2024-03-20,438.79,Paid
8,698,9,2024-05-05,1064.50,Paid
9,888,10,2024-06-21,979.08,Partially Paid


**Pros:** one pass, one readable query — and the *filter condition itself*
never has to touch a date comparison, so it works identically regardless of
what "latest" means (max date, or a tiebreaker on a second column via
`ORDER BY service_date DESC, claim_id DESC`). **Cons vs. the alternative**
(`GROUP BY member_id, MAX(service_date)` then join back to `claims` to get
the rest of the row): the `GROUP BY` + self-join approach needs a second
pass over the data and a join, and gets genuinely awkward the moment you
have a tiebreak rule — the window-function version stays a one-liner change.

### 7.3 PARTITION BY with an aggregate — row + group, together

This is the mechanic the whole fraud section in Part 9 is built on: compute
a group aggregate (`AVG`) but keep every row so you can compare each row to
its own group.

In [27]:
con.execute('''
    SELECT
        provider_id,
        claim_id,
        claim_amount,
        AVG(claim_amount) OVER (PARTITION BY provider_id) AS provider_avg_claim
    FROM claims
    ORDER BY provider_id, claim_amount DESC
    LIMIT 10
''').fetchdf()

,provider_id,claim_id,claim_amount,provider_avg_claim
0,1,230,7101.41,1432.681852
1,1,246,6110.32,1432.681852
2,1,63,3985.82,1432.681852
3,1,507,3800.64,1432.681852
4,1,942,3132.50,1432.681852
5,1,699,3035.93,1432.681852
6,1,939,2915.93,1432.681852
7,1,665,2766.39,1432.681852
8,1,252,2698.34,1432.681852
9,1,620,2290.60,1432.681852


### 7.4 Running totals

Adding `ORDER BY` inside the window frame turns an aggregate into a
*cumulative* one — here, each member's spend-to-date across their claims,
in chronological order.

In [28]:
con.execute('''
    WITH busiest_member AS (
        SELECT member_id FROM claims GROUP BY member_id ORDER BY COUNT(*) DESC LIMIT 1
    )
    SELECT
        member_id,
        service_date,
        paid_amount,
        SUM(paid_amount) OVER (
            PARTITION BY member_id ORDER BY service_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_paid
    FROM claims
    WHERE member_id = (SELECT member_id FROM busiest_member) AND paid_amount IS NOT NULL
    ORDER BY service_date
''').fetchdf()

,member_id,service_date,paid_amount,cumulative_paid
0,138,2023-01-21,1011.89,1011.89
1,138,2023-02-18,0.00,1011.89
2,138,2023-03-20,0.00,1011.89
3,138,2023-03-25,532.64,1544.53
4,138,2023-04-12,1737.67,3282.20
5,138,2023-09-05,0.00,3282.20
6,138,2023-12-08,240.79,3522.99
7,138,2024-03-09,5739.87,9262.86
8,138,2024-03-25,1131.92,10394.78
9,138,2024-03-29,672.86,11067.64


### 7.5 LAG / LEAD

`LAG` looks back N rows within the partition, `LEAD` looks forward. Useful
for period-over-period comparisons or, as here, the **gap between a
member's consecutive pharmacy claims** — a simplified proxy for the kind of
medication-adherence gap analysis payers care about (a real Proportion of
Days Covered metric would also need days-supply-per-fill, which this
schema doesn't model, but the gap pattern is the same building block).

In [29]:
con.execute('''
    WITH pharmacy_claims AS (
        SELECT member_id, claim_id, service_date
        FROM claims
        WHERE claim_type = 'Pharmacy'
    )
    SELECT
        member_id,
        claim_id,
        service_date,
        LAG(service_date) OVER (PARTITION BY member_id ORDER BY service_date) AS previous_fill_date,
        service_date - LAG(service_date) OVER (PARTITION BY member_id ORDER BY service_date) AS days_since_previous_fill
    FROM pharmacy_claims
    ORDER BY member_id, service_date
    LIMIT 15
''').fetchdf()

,member_id,claim_id,service_date,previous_fill_date,days_since_previous_fill
0,1,406,2023-04-23,NaT,<NA>
1,1,656,2024-02-16,2023-04-23,299
2,2,900,2023-04-07,NaT,<NA>
3,2,523,2024-01-26,2023-04-07,294
4,4,176,2024-01-08,NaT,<NA>
5,4,376,2024-05-25,2024-01-08,138
6,6,643,2023-03-27,NaT,<NA>
7,6,636,2023-05-08,2023-03-27,42
8,6,944,2023-06-20,2023-05-08,43
9,6,188,2024-03-30,2023-06-20,284


### 7.6 NTILE — quantile buckets

Splits a partition into N roughly-equal-sized buckets by rank. A common use:
segmenting members into spend quartiles for a risk/utilization program.

In [30]:
con.execute('''
    WITH member_spend AS (
        SELECT member_id, SUM(paid_amount) AS total_paid
        FROM claims
        WHERE paid_amount IS NOT NULL
        GROUP BY member_id
    )
    SELECT
        member_id,
        total_paid,
        NTILE(4) OVER (ORDER BY total_paid DESC) AS spend_quartile   -- 1 = highest spenders
    FROM member_spend
    ORDER BY total_paid DESC
    LIMIT 10
''').fetchdf()

,member_id,total_paid,spend_quartile
0,89,28050.65,1
1,61,20158.23,1
2,2,20152.04,1
3,10,15696.50,1
4,76,15472.58,1
5,90,14057.40,1
6,133,13273.39,1
7,123,13182.18,1
8,58,13074.43,1
9,34,12185.68,1


**Pros (7.3 - 7.6):** window functions eliminate the self-joins and
correlated subqueries these patterns used to require, *and* they keep
row-level detail that a plain `GROUP BY` would throw away. **Cons:** they
can be more computationally expensive than a simple aggregate on very large
partitions (the engine has to materialize per-row results, not just one row
per group), and stacking many window functions in one query can hurt
readability if you're not naming things clearly with CTEs — see Part 9 for
where that discipline starts to matter.

## Part 8 — Expert Techniques

### 8.1 Recursive CTE — walking a hierarchy of arbitrary depth

The self-join in 3.3 only gets you one level up. A `WITH RECURSIVE` CTE
walks the full chain: start from an "anchor" (physicians with no
supervisor), then repeatedly join back to the CTE itself until nothing new
matches.

In [31]:
con.execute('''
    WITH RECURSIVE org_chart AS (
        -- anchor: department heads, no supervisor
        SELECT physician_id, first_name, last_name, supervisor_id, 1 AS level
        FROM physicians
        WHERE supervisor_id IS NULL

        UNION ALL

        -- recursive step: find physicians reporting to anyone already in org_chart
        SELECT p.physician_id, p.first_name, p.last_name, p.supervisor_id, oc.level + 1
        FROM physicians p
        JOIN org_chart oc ON p.supervisor_id = oc.physician_id
    )
    SELECT level, first_name, last_name
    FROM org_chart
    ORDER BY level, last_name
''').fetchdf()

,level,first_name,last_name
0,1,Jason,Baker
1,1,Christine,Barnes
2,1,William,Davis
3,1,Wendy,Rice
4,2,Sara,Allison
5,2,Tricia,Baker
6,2,Joseph,Brennan
7,2,David,Brewer
8,2,Michelle,Brown
9,2,Eric,Campbell


**Pros:** the only clean, set-based way to traverse a hierarchy or graph of
unknown depth — orgs charts, bill-of-materials, category trees. **Cons:**
not supported in every SQL dialect/version (e.g. older MySQL), a missing or
wrong anchor/termination condition can infinite-loop on a real database, and
for a *cyclic* graph (not a clean tree) you need extra cycle-guard logic.
For deep graph algorithms (shortest path, centrality) a Python graph library
like `networkx` is often a better fit than pushing the algorithm into SQL —
see Part 10.

### 8.2 PIVOT — rows to columns

Turning category values into their own columns. DuckDB has a native `PIVOT`
statement; most other engines (or older DuckDB queries) use manual
conditional aggregation (`SUM(CASE WHEN ...)`) to do the same thing.

In [32]:
# Native DuckDB PIVOT syntax — concise, DuckDB/Snowflake/SQL-Server-flavored
con.execute('''
    PIVOT claims
    ON claim_status
    USING COUNT(*)
    GROUP BY claim_type
    ORDER BY claim_type
''').fetchdf()

,claim_type,Denied,Paid,Partially Paid,Pending
0,Inpatient,41,155,35,25
1,Outpatient,42,160,25,20
2,Pharmacy,35,170,20,30
3,Professional,25,167,23,27


In [33]:
# Portable equivalent — verbose, but runs on essentially any SQL engine
con.execute('''
    SELECT
        claim_type,
        SUM(CASE WHEN claim_status = 'Paid'           THEN 1 ELSE 0 END) AS paid,
        SUM(CASE WHEN claim_status = 'Denied'          THEN 1 ELSE 0 END) AS denied,
        SUM(CASE WHEN claim_status = 'Pending'         THEN 1 ELSE 0 END) AS pending,
        SUM(CASE WHEN claim_status = 'Partially Paid'  THEN 1 ELSE 0 END) AS partially_paid
    FROM claims
    GROUP BY claim_type
    ORDER BY claim_type
''').fetchdf()

,claim_type,paid,denied,pending,partially_paid
0,Inpatient,155.0,41.0,25.0,35.0
1,Outpatient,160.0,42.0,20.0,25.0
2,Pharmacy,170.0,35.0,30.0,20.0
3,Professional,167.0,25.0,27.0,23.0


**Pros:** native `PIVOT` is short and self-documenting. **Cons:** it's not
standard SQL and syntax differs across engines that even have it (DuckDB vs.
Snowflake vs. SQL Server); `SUM(CASE WHEN ...)` is uglier but works
*everywhere*, including SQLite/Postgres/MySQL, and is what you'd typically
write on a whiteboard if the target engine isn't specified.

### 8.3 Set operations — UNION / UNION ALL / INTERSECT / EXCEPT

These combine the results of two queries with the *same columns*.

In [34]:
high_value = "SELECT claim_id FROM claims WHERE claim_amount > 20000"
denied     = "SELECT claim_id FROM claims WHERE claim_status = 'Denied'"

print("UNION (either condition, de-duplicated):")
print(con.execute(f"{high_value} UNION {denied}").fetchdf().shape)

print("INTERSECT (both conditions — high-value AND denied, worth a closer look):")
print(con.execute(f"{high_value} INTERSECT {denied}").fetchdf().shape)

print("EXCEPT (high-value but NOT denied — i.e., paid out despite being large):")
print(con.execute(f"{high_value} EXCEPT {denied}").fetchdf().shape)

UNION (either condition, de-duplicated):
(143, 1)
INTERSECT (both conditions — high-value AND denied, worth a closer look):
(0, 1)
EXCEPT (high-value but NOT denied — i.e., paid out despite being large):
(0, 1)


**Pros:** a clean, declarative way to express "either/both/only-one-of"
across two result sets, without an awkward `OR`/join. **Cons:** `UNION`
de-duplicates, which means a sort/hash pass over the combined result — if
you know there's no overlap (or don't care about duplicates), `UNION ALL`
is strictly cheaper and should be your default unless you specifically need
de-duplication.

### 8.4 QUALIFY — filtering on a window function directly

`QUALIFY` is `HAVING` for window functions: it lets you filter on a window
function's result **without** wrapping the query in a subquery/CTE just to
reference it in `WHERE`. (Supported in DuckDB and Snowflake; **not**
standard SQL — Postgres, MySQL, and SQLite don't have it.)

In [35]:
# Same idea as 7.2 (top-N per group), no CTE needed:
con.execute('''
    SELECT provider_id, claim_id, claim_amount
    FROM claims
    QUALIFY ROW_NUMBER() OVER (PARTITION BY provider_id ORDER BY claim_amount DESC) <= 3
    ORDER BY provider_id, claim_amount DESC
''').fetchdf()

,provider_id,claim_id,claim_amount
0,1,230,7101.41
1,1,246,6110.32
2,1,63,3985.82
3,2,622,13038.14
4,2,531,5416.07
5,2,800,4295.52
6,3,324,18752.68
7,3,995,9641.95
8,3,559,8182.75
9,4,106,7220.29


**Pros:** noticeably more concise than the CTE + `WHERE` version for
top-N-per-group and similar patterns. **Cons:** it only works where it's
supported (DuckDB, Snowflake, BigQuery-ish dialects) — writing `QUALIFY` in
an interview or on an engine that doesn't support it is a bug, not a style
choice, so know your target engine before reaching for it. It also only
accepts a window-function expression directly (or column aliases defined in
the *same* `SELECT`) — you can't `QUALIFY` on a value that was only computed
inside an earlier CTE.

### 8.5 EXPLAIN — reading a query plan

`EXPLAIN` shows how the engine intends to execute a query, without running
it. Useful for sanity-checking that filters are actually being applied
early, joins are in a sensible order, etc.

In [36]:
plan = con.execute('''
    EXPLAIN
    SELECT p.provider_name, SUM(c.paid_amount)
    FROM claims c
    JOIN providers p ON p.provider_id = c.provider_id
    WHERE c.claim_status = 'Paid'
    GROUP BY p.provider_name
''').fetchall()

print(plan[0][1])

┌───────────────────────────┐
│       HASH_GROUP_BY       │
│    ────────────────────   │
│         Groups: #0        │
│    Aggregates: sum(#1)    │
│                           │
│         ~181 rows         │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│       provider_name       │
│        paid_amount        │
│                           │
│         ~200 rows         │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         HASH_JOIN         │
│    ────────────────────   │
│      Join Type: INNER     │
│                           │
│        Conditions:        ├──────────────┐
│ provider_id = provider_id │              │
│                           │              │
│         ~200 rows         │              │
└─────────────┬─────────────┘              │
┌─────────────┴─────────────┐┌─────────────┴─────────────┐
│         PROJECTION        ││        PANDAS_SCAN        │
│    ────────────────────  

**What to look for, generally (across engines, not just DuckDB):** filters
pushed down close to the table scan (not applied only after a big join),
joins ordered so the smallest/most-filtered table drives the join, and no
full scan where an index or statistics could have narrowed things early. A
query that reads fine in isolation can still be slow — `EXPLAIN` is how you
find out *why*, instead of guessing.

## Part 9 — Fraud & Anomaly Detection

This is the pattern you started with:

```sql
SELECT
    provider_id,
    claim_id,
    claim_amount,
    AVG(claim_amount) OVER (PARTITION BY provider_id) AS provider_avg_claim,
    CASE
        WHEN claim_amount > 2 * AVG(claim_amount) OVER (PARTITION BY provider_id)
        THEN 1
        ELSE 0
    END AS is_over_2x_avg
FROM claims;
```

The core idea — a window function computes each provider's average without
collapsing the rows, so every claim can be compared to its own peer group in
the same row — is exactly right, and it's the backbone of every query below.
What changes as we go is *how* we define "unusual." Each version fixes a
specific weakness in the one before it.

### 9.1 The starting point: a fixed multiplier

Flag any claim more than 2x its provider's average.

In [37]:
con.execute('''
    WITH flagged AS (
        SELECT
            provider_id,
            claim_id,
            claim_amount,
            AVG(claim_amount) OVER (PARTITION BY provider_id) AS provider_avg_claim,
            CASE
                WHEN claim_amount > 2 * AVG(claim_amount) OVER (PARTITION BY provider_id)
                THEN 1 ELSE 0
            END AS is_over_2x_avg
        FROM claims
    )
    SELECT * FROM flagged
    WHERE is_over_2x_avg = 1
    ORDER BY claim_amount DESC
''').fetchdf()

,provider_id,claim_id,claim_amount,provider_avg_claim,is_over_2x_avg
0,3,324,18752.68,1825.553897,1
1,5,900,16652.60,1699.118251,1
2,5,780,14767.30,1699.118251,1
3,5,637,13186.06,1699.118251,1
4,2,622,13038.14,1584.610962,1
...,...,...,...,...,...
98,6,371,3080.34,1402.752500,1
99,1,699,3035.93,1432.681852,1
100,6,758,2920.76,1402.752500,1
101,1,939,2915.93,1432.681852,1


**Pros:** dead simple, transparent, easy to explain to a non-technical
compliance reviewer ("more than double the provider's typical claim").
**Cons:** the threshold is arbitrary and ignores how *spread out* each
provider's claims naturally are. A provider whose claims are always wildly
variable will trip this constantly on completely normal claims (false
positives); a provider whose claims barely vary might let a genuinely
unusual claim slide under 2x (false negatives).

### 9.2 A statistically-grounded version: z-score

Instead of a flat multiplier, measure how many standard deviations a claim
is from its provider's mean. This adapts to each provider's own natural
variability instead of applying the same rule to everyone.

In [38]:
con.execute('''
    WITH provider_stats AS (
        SELECT
            provider_id,
            claim_id,
            claim_amount,
            AVG(claim_amount)        OVER (PARTITION BY provider_id) AS provider_avg,
            STDDEV_SAMP(claim_amount) OVER (PARTITION BY provider_id) AS provider_stddev
        FROM claims
    ),
    scored AS (
        SELECT
            *,
            (claim_amount - provider_avg) / NULLIF(provider_stddev, 0) AS z_score
        FROM provider_stats
    )
    SELECT provider_id, claim_id, claim_amount, provider_avg, provider_stddev, z_score
    FROM scored
    WHERE ABS(z_score) > 3
    ORDER BY ABS(z_score) DESC
''').fetchdf()

,provider_id,claim_id,claim_amount,provider_avg,provider_stddev,z_score
0,3,324,18752.68,1825.553897,2100.331899,8.059262
1,2,622,13038.14,1584.610962,1492.392564,7.674609
2,5,900,16652.60,1699.118251,2074.593753,7.207908
3,5,780,14767.30,1699.118251,2074.593753,6.299152
4,5,637,13186.06,1699.118251,2074.593753,5.536960
5,5,892,12726.26,1699.118251,2074.593753,5.315326
6,6,292,7383.54,1402.752500,1170.189231,5.110958
7,4,106,7220.29,1578.856064,1308.096698,4.312704
8,7,437,9355.51,1640.058544,1828.484394,4.219588
9,1,230,7101.41,1432.681852,1395.390313,4.062468


**Pros:** fixes 9.1's blind spot — the threshold now scales with each
provider's own variability, so it's harder for a naturally noisy provider to
false-positive its way onto the list. **Cons:** needs enough claims per
provider for the standard deviation to mean anything (a provider with 1-2
claims gives a noisy or undefined stddev — note the `NULLIF(..., 0)` guard
against divide-by-zero for a provider with zero variance). It also assumes a
roughly normal distribution; claim amounts in the real world are usually
right-skewed (a long tail of expensive procedures), so "3 standard
deviations" doesn't mean quite what it would under a bell curve — a
percentile-based approach (9.5) sidesteps that assumption entirely.

### 9.3 QUALIFY version — same idea, less scaffolding

A different but related use case for `QUALIFY` (8.4): surfacing each
provider's largest claims for manual review, without a CTE.

In [39]:
con.execute('''
    SELECT provider_id, claim_id, claim_amount
    FROM claims
    QUALIFY claim_amount > 2 * AVG(claim_amount) OVER (PARTITION BY provider_id)
    ORDER BY claim_amount DESC
''').fetchdf()

,provider_id,claim_id,claim_amount
0,3,324,18752.68
1,5,900,16652.60
2,5,780,14767.30
3,5,637,13186.06
4,2,622,13038.14
...,...,...,...
98,6,371,3080.34
99,1,699,3035.93
100,6,758,2920.76
101,1,939,2915.93


**Note:** `QUALIFY` needs the window function written out directly (or as a
`SELECT`-list alias) — it can't reach into an earlier CTE's already-computed
column the way 9.2's `WHERE` can. That's why 9.2 uses a CTE + `WHERE`
(portable, works with derived stats like `z_score`) while this one inlines
the window function straight into `QUALIFY` (concise, DuckDB/Snowflake
only). Pick based on which constraint matters more for your situation.

### 9.4 Peer-group comparison: specialty instead of provider

9.1-9.3 all compare a claim to *its own provider's* average. That misses a
scenario real fraud analysts care about: if an entire provider is
systematically over-billing, every claim there looks "normal" relative to
its own (also-inflated) peer group. Comparing against the physician's
**specialty** instead exposes that.

In [40]:
con.execute('''
    WITH specialty_stats AS (
        SELECT
            c.claim_id,
            c.provider_id,
            ph.physician_id,
            ph.specialty,
            c.claim_amount,
            AVG(c.claim_amount)        OVER (PARTITION BY ph.specialty) AS specialty_avg,
            STDDEV_SAMP(c.claim_amount) OVER (PARTITION BY ph.specialty) AS specialty_stddev
        FROM claims c
        JOIN physicians ph ON ph.physician_id = c.physician_id
    )
    SELECT
        claim_id, provider_id, physician_id, specialty, claim_amount, specialty_avg,
        (claim_amount - specialty_avg) / NULLIF(specialty_stddev, 0) AS z_score
    FROM specialty_stats
    WHERE (claim_amount - specialty_avg) / NULLIF(specialty_stddev, 0) > 3
    ORDER BY z_score DESC
    LIMIT 15
''').fetchdf()

,claim_id,provider_id,physician_id,specialty,claim_amount,specialty_avg,z_score
0,324,3,10,Cardiology,18752.68,1739.279333,7.732324
1,622,2,15,Family Medicine,13038.14,1694.533833,6.477156
2,637,5,7,Dermatology,13186.06,1620.461881,6.167456
3,900,5,19,Pediatrics,16652.60,2116.268404,5.469366
4,437,7,20,Radiology,9355.51,1517.219080,5.444783
5,853,7,17,Psychiatry,8942.77,1526.494894,5.091313
6,892,5,3,Cardiology,12726.26,1739.279333,4.993411
7,13,5,1,General Surgery,8208.35,1517.594010,4.979918
8,890,7,20,Radiology,8491.10,1517.219080,4.844330
9,780,5,19,Pediatrics,14767.30,2116.268404,4.760012


**Pros:** catches provider-level or physician-level patterns that a
provider-only baseline structurally can't see, because the whole comparison
group is drawn from outside that provider. **Cons:** cross-provider
comparison mixes populations that may legitimately differ in acuity/case
mix — a cardiologist at a trauma center and one at a routine outpatient
clinic don't have the same "normal," so a real system would need to adjust
for case mix (e.g., partition by specialty *and* diagnosis category, not
specialty alone) rather than treat every physician in a specialty as
interchangeable.

### 9.5 Percentile-based: distribution-free outliers

Instead of assuming a shape for the distribution, just rank claims within
their specialty and flag the top slice. `PERCENT_RANK` gives each row's
relative position (0 to 1) within its partition.

In [41]:
con.execute('''
    WITH ranked AS (
        SELECT
            c.claim_id, ph.specialty, c.claim_amount,
            PERCENT_RANK() OVER (PARTITION BY ph.specialty ORDER BY c.claim_amount) AS pct_rank
        FROM claims c
        JOIN physicians ph ON ph.physician_id = c.physician_id
    )
    SELECT * FROM ranked
    WHERE pct_rank >= 0.99          -- top 1% by amount, within specialty
    ORDER BY specialty, claim_amount DESC
''').fetchdf()

,claim_id,specialty,claim_amount,pct_rank
0,324,Cardiology,18752.68,1.000000
1,892,Cardiology,12726.26,0.993289
2,637,Dermatology,13186.06,1.000000
3,255,Dermatology,9365.16,0.990000
4,622,Family Medicine,13038.14,1.000000
5,13,General Surgery,8208.35,1.000000
6,230,General Surgery,7101.41,0.994764
7,292,Internal Medicine,7383.54,1.000000
8,180,Orthopedics,6348.04,1.000000
9,900,Pediatrics,16652.60,1.000000


**Pros:** makes no assumption about the shape of the distribution — robust
to the right-skew that breaks the normality assumption behind z-scores.
**Cons:** by construction, a "top 1%" always exists, even in a population
with zero actual fraud — this tool tells you relative rank, not whether
something is genuinely anomalous. Best used as a **triage/prioritization**
signal (which claims to look at first) rather than a standalone fraud
determination.

### 9.6 Capstone: a multi-signal composite score

Real rules-based fraud triage rarely relies on one signal. Here we combine
three independent flags — amount z-score, unusually high monthly claim
volume for a physician, and an elevated partial-payment rate — into a
simple additive risk score, then surface the claims where multiple signals
agree.

In [42]:
con.execute('''
    WITH amount_signal AS (
        SELECT
            claim_id, physician_id, provider_id, claim_amount, service_date, claim_status,
            CASE WHEN (claim_amount - AVG(claim_amount) OVER (PARTITION BY provider_id))
                      / NULLIF(STDDEV_SAMP(claim_amount) OVER (PARTITION BY provider_id), 0) > 3
                 THEN 1 ELSE 0 END AS amount_flag
        FROM claims
    ),
    volume_signal AS (
        SELECT
            *,
            COUNT(*) OVER (PARTITION BY physician_id, DATE_TRUNC('month', service_date)) AS physician_monthly_claims,
            CASE WHEN COUNT(*) OVER (PARTITION BY physician_id, DATE_TRUNC('month', service_date)) > 5
                 THEN 1 ELSE 0 END AS volume_flag
        FROM amount_signal
    ),
    payment_signal AS (
        SELECT
            *,
            AVG(CASE WHEN claim_status = 'Partially Paid' THEN 1.0 ELSE 0 END)
                OVER (PARTITION BY physician_id) AS physician_partial_rate,
            CASE WHEN AVG(CASE WHEN claim_status = 'Partially Paid' THEN 1.0 ELSE 0 END)
                          OVER (PARTITION BY physician_id) > 0.15
                 THEN 1 ELSE 0 END AS payment_flag
        FROM volume_signal
    )
    SELECT
        claim_id, physician_id, provider_id, claim_amount,
        amount_flag, volume_flag, payment_flag,
        amount_flag + volume_flag + payment_flag AS risk_score
    FROM payment_signal
    WHERE amount_flag + volume_flag + payment_flag >= 2
    ORDER BY risk_score DESC, claim_amount DESC
''').fetchdf()

,claim_id,physician_id,provider_id,claim_amount,amount_flag,volume_flag,payment_flag,risk_score
0,324,10,3,18752.68,1,1,0,2
1,255,12,5,9365.16,1,0,1,2
2,175,6,4,5770.06,1,1,0,2


**Pros:** requiring 2+ independent signals to agree is far more precise than
any single signal alone (9.1-9.5 individually will each flag plenty of
claims that are just normal variation) — this is a realistic sketch of how a
production rules-engine does cheap, fully auditable, explainable triage over
the *entire* dataset before anything gets escalated to a human investigator
or a machine-learning model. **Cons:** it's still a rules engine — every
threshold (`> 3`, `> 5`, `> 0.15`) is a hand-picked constant with no
statistical guarantee, thresholds interact in ways that are hard to reason
about as you add more signals, and it can't learn from labeled outcomes the
way a supervised model could. This is exactly the handoff point between SQL
and Python explored next.

## Part 10 — SQL vs. Python: choosing the right tool

You're more comfortable in Python, so it's worth being explicit about where
each language actually earns its keep, using this same dataset as evidence
rather than opinion. DuckDB runs SQL directly against the pandas DataFrames
we already have in memory (`tables['claims']`, etc.) — so this is a genuinely
fair, apples-to-apples comparison, not "different systems, different data."

### 10.1 Where they're basically a tie: a single group-relative calculation

Reproduce 9.1's flag in pure pandas using `groupby().transform()` — pandas'
`transform` is, mechanically, the same idea as `AVG(...) OVER (PARTITION BY
...)`: compute a group aggregate, broadcast it back to every row in the
group without collapsing them.

In [43]:
claims_df = tables['claims'].copy()
claims_df['provider_avg_claim'] = claims_df.groupby('provider_id')['claim_amount'].transform('mean')
claims_df['is_over_2x_avg'] = (claims_df['claim_amount'] > 2 * claims_df['provider_avg_claim']).astype(int)

claims_df[claims_df['is_over_2x_avg'] == 1] \
    .sort_values('claim_amount', ascending=False) \
    [['provider_id', 'claim_id', 'claim_amount', 'provider_avg_claim', 'is_over_2x_avg']] \
    .head(10)

,provider_id,claim_id,claim_amount,provider_avg_claim,is_over_2x_avg
323,3,324,18752.68,1825.553897,1
899,5,900,16652.60,1699.118251,1
779,5,780,14767.30,1699.118251,1
636,5,637,13186.06,1699.118251,1
621,2,622,13038.14,1584.610962,1
891,5,892,12726.26,1699.118251,1
994,3,995,9641.95,1825.553897,1
254,5,255,9365.16,1699.118251,1
436,7,437,9355.51,1640.058544,1
852,7,853,8942.77,1640.058544,1


For *this specific operation* the two are nearly identical in length and
readability. The real differentiators show up in the next two examples.

### 10.2 Where SQL shines: arbitrary-depth, set-based traversal

Compare the 6-line recursive CTE from 8.1 to a hand-rolled Python walk of
the same hierarchy. The logic is more code, and — unlike the CTE — it's a
sequential loop you have to get right yourself (including not infinite-
looping on a cycle).

In [44]:
physicians_df = tables['physicians']

by_id = physicians_df.set_index('physician_id')
children = {}
for pid, row in by_id.iterrows():
    sup = row['supervisor_id']
    if pd.notna(sup):
        children.setdefault(int(sup), []).append(pid)

rows = []
frontier = [(pid, 1) for pid in by_id[by_id['supervisor_id'].isna()].index]
while frontier:
    pid, level = frontier.pop(0)
    rows.append({'level': level, 'physician_id': pid,
                 'name': f"{by_id.loc[pid, 'first_name']} {by_id.loc[pid, 'last_name']}"})
    for child_id in children.get(pid, []):
        frontier.append((child_id, level + 1))

pd.DataFrame(rows).sort_values(['level', 'name'])

,level,physician_id,name
2,1,16,Christine Barnes
3,1,18,Jason Baker
1,1,2,Wendy Rice
0,1,1,William Davis
6,2,9,Barbara Sanchez
19,2,14,Beth Daniels
17,2,5,Brian Humphrey
12,2,15,Christopher Lopez
13,2,17,David Brewer
4,2,7,David Walker


**Verdict:** SQL's recursive CTE is declarative — you describe the anchor
and the recursive step, and the engine handles traversal order and
termination. The pandas version works, but *you* are now responsible for
the queue, the cycle safety, and the bookkeeping. For a graph problem
that's genuinely complex (shortest paths, centrality, cycles), reach for a
dedicated Python graph library (`networkx`) rather than either of these —
but for a straightforward tree walk on data that's already in a table,
SQL's recursive CTE is usually less code and less to get wrong.

### 10.3 Where Python shines: multivariate statistics / ML

Every query in Part 9 flags a claim based on *one* dimension at a time
(amount, or volume, or payment pattern), combined only by simple addition.
A real anomaly-detection model can reason about several features
*jointly* — this is squarely outside what SQL is built for, and squarely
inside what a library like `scikit-learn` is built for.

We'll use SQL to do what it's good at (shaping and aggregating the raw
claims/physician data into a clean feature table), then hand that
much-smaller table to Python for the statistical layer.

In [45]:
from sklearn.ensemble import IsolationForest

physician_features = con.execute('''
    SELECT
        physician_id,
        COUNT(*)                                                    AS n_claims,
        AVG(claim_amount)                                           AS avg_amount,
        STDDEV_SAMP(claim_amount)                                   AS stddev_amount,
        AVG(DATEDIFF('day', service_date, submitted_date))          AS avg_days_to_submit,
        AVG(CASE WHEN claim_status = 'Denied' THEN 1.0 ELSE 0 END)  AS denial_rate
    FROM claims
    GROUP BY physician_id
''').fetchdf().fillna(0)

feature_cols = ['n_claims', 'avg_amount', 'stddev_amount', 'avg_days_to_submit', 'denial_rate']
model = IsolationForest(n_estimators=200, contamination=0.15, random_state=42)
physician_features['anomaly_score'] = model.fit_predict(physician_features[feature_cols])
# IsolationForest labels: -1 = anomaly, 1 = normal

physician_features.sort_values('anomaly_score').head(10)

,physician_id,n_claims,avg_amount,stddev_amount,avg_days_to_submit,denial_rate,anomaly_score
3,2,44,1434.716136,1042.475038,8.909091,0.227273,-1
7,5,33,1824.010909,1342.140633,11.666667,0.060606,-1
12,19,47,2415.090426,3323.477298,10.787234,0.085106,-1
1,4,57,1471.672105,1227.158175,10.614035,0.052632,1
2,10,51,2083.515294,2806.892853,10.176471,0.117647,1
4,18,54,1451.007778,1230.091703,10.000000,0.129630,1
5,14,59,1394.066102,1270.024139,9.898305,0.084746,1
0,1,53,1589.620566,1536.150439,10.056604,0.188679,1
6,17,50,1607.260200,1749.073584,10.300000,0.200000,1
8,9,47,1556.750426,1185.373445,9.702128,0.212766,1


**Verdict:** this is not a query you can write in SQL — `IsolationForest` is
reasoning about five features *jointly* (a physician with a slightly high
denial rate *and* slightly high amounts *and* a slow submission pattern
might be more suspicious in combination than any single feature alone would
suggest), which needs a model, not a threshold. Notice the pipeline
structure: **SQL did the aggregation** (raw claims -> one row per physician),
and **Python did the statistics** on the resulting small table. That split
is the realistic default, not an either/or choice.

### 10.4 Decision guide

| Reach for **SQL** when... | Reach for **Python** when... |
|---|---|
| The transformation is naturally set-based: filtering, joining, grouping, ranking within groups | The logic is genuinely procedural/iterative: graph walks, simulations, custom state machines |
| Data already lives in a database/warehouse — you want to push computation close to the data | You need statistics/ML libraries (`scikit-learn`, `statsmodels`, `scipy`) |
| You want the logic to be portable and auditable by anyone who can read SQL — analysts, other engineers, a BI tool | You're gluing together multiple data sources, doing heavy string/regex cleaning, or building visualizations |
| The dataset doesn't comfortably fit in memory | You want unit tests around business logic, or need version-controlled, testable code paths |

**In practice, the default is both, not either:** use SQL to filter, join,
and aggregate a large raw table down to a small, clean, analysis-ready
result — then use Python for anything statistical, iterative, or ML-shaped
on top of that much smaller result. Parts 9 and 10 of this notebook are
that pipeline end to end.

One more wrinkle worth knowing: DuckDB itself blurs this line — you just
saw it run SQL *directly* against pandas DataFrames and hand results right
back as a DataFrame. That's part of why SQL is worth being fluent in even if
you're Python-first: with a tool like this, reaching for a `GROUP BY`
instead of a multi-line `groupby()`/`merge()` chain isn't "using a different
system," it's just picking the more concise way to say the same thing —
and you can drop back into pandas the moment SQL stops being the easy way to
say what you mean.

In [46]:
con.close()